# GLD Data Comparison: Alpha Vantage vs YFinance

This notebook fetches daily financial data for GLD from both Alpha Vantage and Yfinance, compares the results, and visualizes them to help you choose the best source for your workflow.

In [1]:
# 1. Import Required Libraries
import requests
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt


In [2]:
# 2. Fetch Daily Data from Alpha Vantage
ALPHA_AD_API = "74M88OXCGWTNUIV9"  # Replace with your real key if needed
symbol = "GLD"
url = f'https://www.alphavantage.co/query?function=TIME_SERIES_DAILY_ADJUSTED&symbol={symbol}&outputsize=full&apikey={ALPHA_AD_API}'
response = requests.get(url)
data = response.json()
if "Time Series (Daily)" not in data:
    print("Error fetching data:", data)
    alpha_df = None
else:
    ts_daily = data["Time Series (Daily)"]
    alpha_df = pd.DataFrame.from_dict(ts_daily, orient='index')
    alpha_df.rename(columns={
        "1. open": "Open",
        "2. high": "High",
        "3. low": "Low",
        "4. close": "Close",
        "5. adjusted close": "Adj Close",
        "6. volume": "Volume",
        "7. dividend amount": "Dividend",
        "8. split coefficient": "Split Coefficient"
    }, inplace=True)
    alpha_df.index = pd.to_datetime(alpha_df.index)
    for col in ["Open", "High", "Low", "Close", "Adj Close", "Volume", "Dividend", "Split Coefficient"]:
        alpha_df[col] = alpha_df[col].astype(float)
    alpha_df.sort_index(inplace=True)


Error fetching data: {'Information': 'Thank you for using Alpha Vantage! This is a premium endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium endpoints'}


In [3]:
# 3. Fetch Daily Data from Yfinance
yf_df = yf.download("GLD", start="2004-01-01", progress=False)
# Ensure columns match Alpha Vantage
yf_df.rename(columns={
    "Open": "Open",
    "High": "High",
    "Low": "Low",
    "Close": "Close",
    "Adj Close": "Adj Close",
    "Volume": "Volume"
}, inplace=True)
yf_df.index = pd.to_datetime(yf_df.index)
yf_df.sort_index(inplace=True)


/tmp/ipykernel_27879/3922376917.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  yf_df = yf.download("GLD", start="2004-01-01", progress=False)


In [4]:
# 4. Preprocess and Align DataFrames
# Ensure both DataFrames have the same columns and date index
common_cols = ['Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
if alpha_df is not None:
    alpha_df = alpha_df[common_cols]
yf_df = yf_df[common_cols]
# Align on intersection of dates
if alpha_df is not None:
    shared_dates = alpha_df.index.intersection(yf_df.index)
    alpha_df = alpha_df.loc[shared_dates]
    yf_df = yf_df.loc[shared_dates]


KeyError: "['Adj Close'] not in index"

In [ ]:
# 5. Compare Data Sources (Head, Tail, Stats)
if alpha_df is not None:
    print('Alpha Vantage head:')
    display(alpha_df.head())
    print('Alpha Vantage tail:')
    display(alpha_df.tail())
    print('Alpha Vantage stats:')
    display(alpha_df.describe())
print('Yfinance head:')
display(yf_df.head())
print('Yfinance tail:')
display(yf_df.tail())
print('Yfinance stats:')
display(yf_df.describe())
# Compare closing price stats
if alpha_df is not None:
    print('Close price mean (Alpha):', alpha_df['Close'].mean())
print('Close price mean (Yfinance):', yf_df['Close'].mean())


In [ ]:
# 6. Visualize Closing Prices from Both Sources
plt.figure(figsize=(14,6))
if alpha_df is not None:
    plt.plot(alpha_df.index, alpha_df['Close'], label='Alpha Vantage Close', alpha=0.7)
plt.plot(yf_df.index, yf_df['Close'], label='Yfinance Close', alpha=0.7)
plt.title('GLD Closing Prices: Alpha Vantage vs Yfinance')
plt.xlabel('Date')
plt.ylabel('Close Price')
plt.legend()
plt.show()


In [ ]:
# 7. Save Data to CSV Files
if alpha_df is not None:
    alpha_df.to_csv('GLD_daily_alpha.csv')
yf_df.to_csv('GLD_daily_yf.csv')
print('Saved Alpha Vantage data to GLD_daily_alpha.csv')
print('Saved Yfinance data to GLD_daily_yf.csv')
